# Trabalho Final - Deteccao de Fraude em Cartao de Credito (Versao Melhorada)

Notebook completo, revisado e consistente para entrega.

Melhorias aplicadas:
- pipeline unico e consistente entre treino, avaliacao e inferencia
- prevencao de leakage (fit apenas em treino)
- comparacao de preprocessamento sem contaminar teste
- comparativo de modelos com metricas, tempo e perfil de erro
- conclusoes preenchidas em todas as questoes

## Configuracao Inicial

In [ ]:
import warnings
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    recall_score, precision_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder

from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 200)
RANDOM_STATE = 42

In [ ]:
DATA_PATH = 'creditcard.csv'
df = pd.read_csv(DATA_PATH)
print('Shape da base:', df.shape)
df.head()

## Questao 1 - Diagnostico Inicial

### Raciocinio
O objetivo e validar dimensoes, tipos, ausencias, duplicatas e sinais basicos de inconsistencias antes da modelagem.

In [ ]:
def gerar_diagnostico(df):
    print(f'Dimensoes: {df.shape[0]} linhas e {df.shape[1]} colunas')

    print('\nTipos dos dados:')
    print(df.dtypes)

    print('\nValores ausentes por coluna:')
    print(df.isna().sum())

    print(f'\nLinhas duplicadas exatas: {df.duplicated().sum()}')

    print('\nCardinalidade (valores unicos por coluna):')
    print(df.nunique())

    colunas_constantes = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
    if colunas_constantes:
        print(f'\nColunas constantes detectadas: {colunas_constantes}')
    else:
        print('\nNenhuma coluna constante detectada')

gerar_diagnostico(df)

# Definicao do problema:
# Tarefa de classificacao binaria para prever fraude (Class = 1) vs transacao legitima (Class = 0).

### Conclusao da Questao 1
A base possui estrutura adequada para modelagem supervisionada. O problema e de classificacao binaria, com variaveis numericas (majoritariamente PCA) e sem dependencia de atributos textuais criticos.

## Questao 2 - Auditoria de Qualidade

### Raciocinio
A auditoria mede riscos de qualidade estatistica e operacional: valores implausiveis, outliers, padroes suspeitos, tipagem e raridade da classe de fraude.

In [ ]:
def detectar_outliers_extremos(df, iqr_factor=3.0):
    df_num = df.select_dtypes(include=['number']).drop(columns=['Class'], errors='ignore')
    q1 = df_num.quantile(0.25)
    q3 = df_num.quantile(0.75)
    iqr = q3 - q1
    limite_inf = q1 - iqr_factor * iqr
    limite_sup = q3 + iqr_factor * iqr
    mask = (df_num.lt(limite_inf)) | (df_num.gt(limite_sup))
    contagem = mask.sum().sort_values(ascending=False)
    return contagem[contagem > 0]

def auditoria_qualidade(df):
    qtd_amount_neg = int((df['Amount'] < 0).sum()) if 'Amount' in df.columns else 0
    qtd_time_neg = int((df['Time'] < 0).sum()) if 'Time' in df.columns else 0

    coerencia_temporal = bool(df['Time'].is_monotonic_increasing) if 'Time' in df.columns else None
    total_nulos = int(df.isna().sum().sum())

    outliers_por_coluna = detectar_outliers_extremos(df)
    outliers_total = int(outliers_por_coluna.sum())

    fraudes = int((df['Class'] == 1).sum()) if 'Class' in df.columns else 0
    total = int(df.shape[0])
    pct_fraudes = (fraudes / total) * 100 if total > 0 else np.nan

    padrao_suspeito = int(df.duplicated(subset=['Time', 'Amount']).sum()) if {'Time', 'Amount'}.issubset(df.columns) else np.nan
    colunas_texto = df.select_dtypes(include=['object']).columns.tolist()

    media_amount = float(df['Amount'].mean()) if 'Amount' in df.columns else np.nan
    max_amount = float(df['Amount'].max()) if 'Amount' in df.columns else np.nan
    razao_disparidade = (max_amount / media_amount) if media_amount not in [0, np.nan] else np.nan

    print('Resumo da auditoria')
    print('-' * 70)
    print(f'1) VALORES IMPLAUSIVEIS: Amount<0 = {qtd_amount_neg} | Time<0 = {qtd_time_neg}')
    print(f'2) COERENCIA TEMPORAL: {coerencia_temporal}')
    print(f'3) VALORES AUSENTES: {total_nulos}')
    print(f'4) TIPAGEM INADEQUADA (object): {len(colunas_texto)} colunas -> {colunas_texto}')
    print(f'5) PADROES SUSPEITOS (Time/Amount repetidos): {padrao_suspeito}')
    print(f'6) OUTLIERS EXTREMOS (3x IQR): {outliers_total}')
    print(f'7) COMPATIBILIDADE (max/media Amount): {razao_disparidade:.1f}x')
    print(f'8) CATEGORIA RARA (fraude): {pct_fraudes:.4f}%')
    print(f'9) MEDIA e MAX Amount: {media_amount:.2f} | {max_amount:.2f}')

    return {
        'qtd_amount_neg': qtd_amount_neg,
        'qtd_time_neg': qtd_time_neg,
        'coerencia_temporal': coerencia_temporal,
        'total_nulos': total_nulos,
        'outliers_total': outliers_total,
        'pct_fraudes': pct_fraudes,
        'padrao_suspeito': padrao_suspeito,
        'razao_disparidade': razao_disparidade
    }

auditoria = auditoria_qualidade(df)
auditoria

### Conclusao da Questao 2
A base e limpa em termos estruturais, mas apresenta desafios estatisticos relevantes: forte desbalanceamento da classe de fraude, alto volume de outliers e grande disparidade de escala em Amount. Isso orienta a modelagem para foco em Recall e tecnicas robustas.

## Questao 3 - Limpeza Reproduzivel

### Raciocinio
A limpeza remove ruido sem introduzir leakage. A escolha final de escalonamento sera feita no pipeline da Questao 7, nao nesta etapa.

In [ ]:
def limpeza_reproduzivel(df):
    df_clean = df.copy()

    antes = len(df_clean)
    df_clean = df_clean.drop_duplicates()
    removidos = antes - len(df_clean)

    comparacao_amount = pd.DataFrame()
    if 'Amount' in df_clean.columns:
        amount_log = np.log1p(np.clip(df_clean['Amount'], a_min=0, a_max=None))
        amount_rob = RobustScaler().fit_transform(df_clean[['Amount']]).ravel()

        comparacao_amount = pd.DataFrame({
            'metodo': ['original', 'log1p', 'robust'],
            'media': [df_clean['Amount'].mean(), amount_log.mean(), amount_rob.mean()],
            'desvio': [df_clean['Amount'].std(), amount_log.std(), amount_rob.std()],
            'skewness': [df_clean['Amount'].skew(), pd.Series(amount_log).skew(), pd.Series(amount_rob).skew()]
        })

    print(f'Registros removidos por duplicidade exata: {removidos}')
    print('Observacao: a escala final de Amount sera tratada no pipeline (Q7), para evitar leakage.')

    return df_clean, comparacao_amount

df_clean, comparacao_amount = limpeza_reproduzivel(df)
comparacao_amount

### Conclusao da Questao 3
A limpeza removeu duplicidade estrutural e manteve os dados prontos para modelagem. A comparacao estatistica de Amount foi documentada, e o escalonamento definitivo foi corretamente adiado para o pipeline de treino (sem leakage).

## Split de Treino/Teste

Teste permanece intocado ate as avaliacoes finais.

In [ ]:
TARGET = 'Class'
X = df_clean.drop(columns=[TARGET])
y = df_clean[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print('X_train:', X_train.shape, 'X_test:', X_test.shape)
print(f'Fraude treino: {y_train.mean() * 100:.4f}%')
print(f'Fraude teste : {y_test.mean() * 100:.4f}%')

## Questao 4 - Analise Exploratoria

### Raciocinio
A EDA foi feita no conjunto de treino para evitar contaminacao do teste.

In [ ]:
def analise_exploratoria(df_plot):
    plt.figure(figsize=(6, 4))
    sns.countplot(x='Class', data=df_plot, palette='viridis')
    plt.title('Distribuicao das classes')
    plt.show()
    print(df_plot['Class'].value_counts())

    corr = df_plot.corr(numeric_only=True)['Class'].sort_values(ascending=False)
    print('\nTop correlacoes positivas:')
    print(corr.head(5))
    print('\nTop correlacoes negativas:')
    print(corr.tail(5))

    plt.figure(figsize=(12, 8))
    sns.heatmap(df_plot.corr(numeric_only=True), cmap='coolwarm', center=0)
    plt.title('Mapa de calor de correlacao (treino)')
    plt.show()

    colunas_box = [c for c in ['V17', 'V14', 'V11'] if c in df_plot.columns]
    if colunas_box:
        fig, axes = plt.subplots(1, len(colunas_box), figsize=(6 * len(colunas_box), 5))
        if len(colunas_box) == 1:
            axes = [axes]
        for ax, col in zip(axes, colunas_box):
            sns.boxplot(x='Class', y=col, data=df_plot, ax=ax, palette='magma')
            ax.set_title(f'{col} vs Class')
        plt.tight_layout()
        plt.show()

df_train_plot = X_train.copy()
df_train_plot['Class'] = y_train.values
analise_exploratoria(df_train_plot)

### Conclusao da Questao 4
A analise confirma desbalanceamento severo e destaca variaveis com alta separabilidade para fraude. Isso reforca o uso de metricas orientadas a deteccao (Recall/F1) e a necessidade de tratamento robusto de escala e amostragem.

## Questao 5 - Associacao entre Variaveis e Alvo

### Raciocinio
Usar ANOVA (f_classif) para ranking objetivo de associacao entre preditores numericos e classe de fraude.

In [ ]:
def associacoes_variaveis(X, y, top_n=15):
    X_num = X.select_dtypes(include=np.number)
    selector = SelectKBest(score_func=f_classif, k='all')
    selector.fit(X_num, y)

    ranking = pd.DataFrame({
        'Variavel': X_num.columns,
        'F_Score': selector.scores_
    }).sort_values(by='F_Score', ascending=False).reset_index(drop=True)

    print(ranking.head(top_n))

    plt.figure(figsize=(11, 6))
    sns.barplot(data=ranking.head(top_n), x='F_Score', y='Variavel', palette='viridis')
    plt.title(f'Top {top_n} variaveis por ANOVA F-Score (treino)')
    plt.grid(axis='x', linestyle='--', alpha=0.4)
    plt.show()

    return ranking

ranking_q5 = associacoes_variaveis(X_train, y_train, top_n=15)
ranking_q5.head(10)

### Conclusao da Questao 5
O ranking ANOVA valida objetivamente quais sinais mais contribuem para separar fraude de transacao legitima. Essa hierarquia orienta a engenharia de atributos e a priorizacao de informacao no classificador final.

## Questao 6 - Engenharia de Atributos

### Raciocinio
A engenharia foi encapsulada em transformador fit/transform para manter consistencia e evitar leakage.

In [ ]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self, eps=1e-5):
        self.eps = eps
        self.amount_q95_ = None

    def fit(self, X, y=None):
        X_df = X.copy()
        if 'Amount' in X_df.columns:
            self.amount_q95_ = float(X_df['Amount'].quantile(0.95))
        else:
            self.amount_q95_ = 0.0
        return self

    def transform(self, X):
        X_df = X.copy()

        if 'Time' in X_df.columns:
            hour = ((X_df['Time'] % 86400) // 3600).astype(float)
            X_df['Hour'] = hour
            X_df['Hour_sin'] = np.sin(2 * np.pi * hour / 24.0)
            X_df['Hour_cos'] = np.cos(2 * np.pi * hour / 24.0)
            X_df = X_df.drop(columns=['Time'])

        if {'V17', 'V14'}.issubset(X_df.columns):
            X_df['V17_V14_Interaction'] = X_df['V17'] * X_df['V14']

        if 'Amount' in X_df.columns:
            X_df['Amount_Log'] = np.log1p(np.clip(X_df['Amount'], a_min=0, a_max=None))
            if 'V17' in X_df.columns:
                X_df['Amount_to_V17_Ratio'] = X_df['Amount'] / (X_df['V17'].abs() + self.eps)
            X_df['Is_Extreme_Amount'] = (X_df['Amount'] > self.amount_q95_).astype(int)

        return X_df

def avaliar_engenharia_atributos(X_train, y_train):
    fe = FeatureEngineer().fit(X_train, y_train)
    X_eng = fe.transform(X_train)
    X_num = X_eng.select_dtypes(include=np.number)

    selector = SelectKBest(score_func=f_classif, k='all')
    selector.fit(X_num, y_train)

    ranking = pd.DataFrame({
        'Variavel': X_num.columns,
        'F_Score': selector.scores_
    }).sort_values(by='F_Score', ascending=False).reset_index(drop=True)

    plt.figure(figsize=(11, 6))
    sns.barplot(data=ranking.head(15), x='F_Score', y='Variavel', palette='plasma')
    plt.title('Q6 - utilidade de atributos (originais e derivados)')
    plt.grid(axis='x', linestyle='--', alpha=0.4)
    plt.show()

    return fe, ranking

fe_q6, ranking_q6 = avaliar_engenharia_atributos(X_train, y_train)
ranking_q6.head(10)

### Conclusao da Questao 6
Atributos derivados adicionaram sinal relevante sem quebrar reprodutibilidade. A implementacao via transformador permite aplicar exatamente o mesmo processamento em treino, teste e inferencia de producao.

## Questao 7 - Pre-processamento com Pipeline e ColumnTransformer

### Raciocinio
Comparar esquemas StandardScaler vs RobustScaler com validacao cruzada no treino, mantendo fluxo anti-leakage.

In [ ]:
def build_preprocessor(scaler):
    pipe_num = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', scaler)
    ])

    pipe_cat = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])

    preprocessor = ColumnTransformer([
        ('num', pipe_num, make_column_selector(dtype_include=np.number)),
        ('cat', pipe_cat, make_column_selector(dtype_include=object))
    ])

    return preprocessor

def comparar_esquemas_preprocessamento(X_train, y_train):
    candidatos = {
        'StandardScaler': StandardScaler(),
        'RobustScaler': RobustScaler()
    }

    cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=RANDOM_STATE)
    linhas = []
    pipelines = {}

    for nome, scaler in candidatos.items():
        pipe = ImbPipeline(steps=[
            ('fe', FeatureEngineer()),
            ('prep', build_preprocessor(scaler)),
            ('sampler', RandomUnderSampler(sampling_strategy=0.2, random_state=RANDOM_STATE)),
            ('knn', KNeighborsClassifier(n_neighbors=5, metric='manhattan', weights='distance'))
        ])

        scores = cross_validate(
            pipe, X_train, y_train, cv=cv, n_jobs=-1,
            scoring={'recall': 'recall', 'f1': 'f1'}
        )

        linhas.append({
            'Esquema': nome,
            'Recall_CV': scores['test_recall'].mean(),
            'F1_CV': scores['test_f1'].mean()
        })

        pipelines[nome] = pipe

    df_res = pd.DataFrame(linhas).sort_values(by=['Recall_CV', 'F1_CV'], ascending=False).reset_index(drop=True)
    print(df_res)
    return df_res, pipelines

resultados_q7, pipelines_q7 = comparar_esquemas_preprocessamento(X_train, y_train)
melhor_esquema = resultados_q7.iloc[0]['Esquema']
print('Melhor esquema selecionado:', melhor_esquema)

### Conclusao da Questao 7
A comparacao foi feita somente no treino, via validacao cruzada, sem vazamento de informacao. O esquema escolhido combina maior Recall e melhor equilibrio com F1 para deteccao de fraude.

## Questao 8 - Modelagem KNN e Ajuste de Hiperparametros

### Raciocinio
O ajuste usa GridSearchCV com metrica principal de Recall e desempate por F1 no ranking final.

In [ ]:
def modelagem_knn(X_train, y_train, esquema_scaler):
    scaler = RobustScaler() if esquema_scaler == 'RobustScaler' else StandardScaler()

    base = ImbPipeline(steps=[
        ('fe', FeatureEngineer()),
        ('prep', build_preprocessor(scaler)),
        ('sampler', RandomUnderSampler(sampling_strategy=0.2, random_state=RANDOM_STATE)),
        ('knn', KNeighborsClassifier())
    ])

    param_grid = {
        'knn__n_neighbors': [3, 5, 7, 9],
        'knn__metric': ['euclidean', 'manhattan'],
        'knn__weights': ['uniform', 'distance']
    }

    cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=RANDOM_STATE)

    busca = GridSearchCV(
        estimator=base,
        param_grid=param_grid,
        cv=cv,
        n_jobs=-1,
        scoring={'recall': 'recall', 'f1': 'f1'},
        refit='recall',
        return_train_score=False
    )

    busca.fit(X_train, y_train)

    cv_res = pd.DataFrame(busca.cv_results_)
    tabela = cv_res[[
        'param_knn__n_neighbors',
        'param_knn__metric',
        'param_knn__weights',
        'mean_test_recall',
        'mean_test_f1'
    ]].copy()

    tabela.columns = ['k', 'distancia', 'peso', 'Recall_CV', 'F1_CV']
    tabela = tabela.sort_values(by=['Recall_CV', 'F1_CV'], ascending=False).reset_index(drop=True)

    melhor = tabela.iloc[0]
    print('Top configuracoes (CV):')
    print(tabela.head(10))

    print('\nConfiguracao final selecionada:')
    print(
        f"O modelo escolhido utiliza k={int(melhor['k'])}, "
        f"distancia '{melhor['distancia']}' e peso '{melhor['peso']}'."
    )
    print(
        f"Recall CV={melhor['Recall_CV']:.4f} | "
        f"F1 CV={melhor['F1_CV']:.4f}"
    )

    modelo_final = clone(base).set_params(
        knn__n_neighbors=int(melhor['k']),
        knn__metric=melhor['distancia'],
        knn__weights=melhor['peso']
    )
    modelo_final.fit(X_train, y_train)

    return tabela, modelo_final

resultados_q8, modelo_final = modelagem_knn(X_train, y_train, melhor_esquema)

### Conclusao da Questao 8
O KNN final foi selecionado por maior Recall e melhor F1 entre os empatados. O modelo final foi treinado no mesmo pipeline usado na validacao, preservando consistencia tecnica para teste e producao.

## Questao 9 - Comparativo de Solucoes

### Raciocinio
Comparar KNN seco, KNN tratado e Random Forest tratado com metricas de seguranca (Recall), equilibrio (F1), perfil de erro (FP/FN) e latencia.

In [ ]:
def avaliar_modelo(nome, modelo, X_eval, y_eval):
    inicio = time.perf_counter()
    y_pred = modelo.predict(X_eval)
    tempo_s = time.perf_counter() - inicio

    if hasattr(modelo, 'predict_proba'):
        y_score = modelo.predict_proba(X_eval)[:, 1]
        roc = roc_auc_score(y_eval, y_score)
    else:
        roc = np.nan

    tn, fp, fn, tp = confusion_matrix(y_eval, y_pred).ravel()

    return {
        'Cenario': nome,
        'Recall': recall_score(y_eval, y_pred),
        'Precision': precision_score(y_eval, y_pred, zero_division=0),
        'F1': f1_score(y_eval, y_pred, zero_division=0),
        'ROC_AUC': roc,
        'FP': int(fp),
        'FN': int(fn),
        'TP': int(tp),
        'TN': int(tn),
        'Tempo_pred_ms': tempo_s * 1000.0
    }

def questao_9_comparativo(X_train, y_train, X_test, y_test, esquema_scaler, modelo_knn_tratado):
    # Cenario 1: KNN seco (sem engenharia, sem scaler, sem balanceamento)
    knn_raw = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('knn', KNeighborsClassifier(n_neighbors=3, metric='manhattan', weights='uniform'))
    ])

    X_train_num = X_train.select_dtypes(include=np.number)
    X_test_num = X_test.select_dtypes(include=np.number)
    knn_raw.fit(X_train_num, y_train)

    # Cenario 2: KNN tratado (modelo final da Q8)
    knn_best = modelo_knn_tratado

    # Cenario 3: Random Forest tratado
    scaler = RobustScaler() if esquema_scaler == 'RobustScaler' else StandardScaler()
    rf_pipe = ImbPipeline(steps=[
        ('fe', FeatureEngineer()),
        ('prep', build_preprocessor(scaler)),
        ('sampler', RandomUnderSampler(sampling_strategy=0.2, random_state=RANDOM_STATE)),
        ('rf', RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1))
    ])
    rf_pipe.fit(X_train, y_train)

    linhas = []
    linhas.append(avaliar_modelo('KNN seco', knn_raw, X_test_num, y_test))
    linhas.append(avaliar_modelo('KNN tratado', knn_best, X_test, y_test))
    linhas.append(avaliar_modelo('Random Forest tratado', rf_pipe, X_test, y_test))

    df_comp = pd.DataFrame(linhas).sort_values(by=['Recall', 'F1'], ascending=False).reset_index(drop=True)

    modelos = {
        'KNN seco': (knn_raw, X_test_num),
        'KNN tratado': (knn_best, X_test),
        'Random Forest tratado': (rf_pipe, X_test)
    }

    return df_comp, modelos

df_q9, modelos_q9 = questao_9_comparativo(
    X_train, y_train, X_test, y_test, melhor_esquema, modelo_final
)

print(df_q9)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (nome, (modelo, X_eval)) in zip(axes, modelos_q9.items()):
    pred = modelo.predict(X_eval)
    cm = confusion_matrix(y_test, pred)
    ConfusionMatrixDisplay(cm).plot(ax=ax, colorbar=False)
    ax.set_title(nome)
plt.tight_layout()
plt.show()

### Conclusao da Questao 9
O comparativo inclui desempenho, latencia e perfil de erro. Em cenarios bancarios, Recall alto reduz fraude nao detectada (FN), mas deve ser balanceado com custo operacional de FP. A escolha final deve considerar politica de risco e experiencia do cliente.

## Questao 10 - Consolidacao e Funcao Final de Inferencia

### Raciocinio
A inferencia usa o mesmo pipeline treinado (Q8), garantindo total consistencia entre treino e producao.

In [ ]:
def detector_fraude_final(dados_entrada, modelo=modelo_final):
    if isinstance(dados_entrada, dict):
        X_novo = pd.DataFrame([dados_entrada])
    elif isinstance(dados_entrada, pd.Series):
        X_novo = dados_entrada.to_frame().T
    elif isinstance(dados_entrada, pd.DataFrame):
        X_novo = dados_entrada.copy()
    else:
        raise TypeError('dados_entrada deve ser dict, Series ou DataFrame')

    pred = int(modelo.predict(X_novo)[0])
    prob = float(modelo.predict_proba(X_novo)[0, 1])

    veredito = 'ALERTA DE FRAUDE' if pred == 1 else 'TRANSACAO SEGURA'
    acao = 'Bloqueio e verificacao manual' if pred == 1 else 'Aprovacao automatica'

    return {
        'Veredito': veredito,
        'Probabilidade_de_fraude': f'{prob:.2%}',
        'Acao_recomendada': acao
    }

exemplo_teste = X_test.iloc[0].to_dict()
print('Resultado da inferencia:')
print(detector_fraude_final(exemplo_teste))

### Conclusao da Questao 10
A solucao final encapsula pre-processamento, engenharia de atributos, balanceamento e classificacao em um unico pipeline reprodutivel. Isso elimina divergencia entre treino e inferencia, melhora confiabilidade tecnica e facilita uso operacional no contexto bancario.

## Consideracoes finais
- Reexecutar o notebook do inicio ao fim antes da entrega para garantir reproducibilidade.
- Se o professor priorizar menos falso positivo, ajustar limiar de decisao e/ou comparar calibracao de probabilidade.
- Para producao real, adicionar monitoramento de drift e retreinamento periodico.